In [ ]:
import os
import shutil
import random
from PIL import Image

work_train_dir = '/kaggle/working/dataset/train'
work_test_dir = '/kaggle/working/dataset/test'

if os.path.exists('/kaggle/working/dataset'):
    shutil.rmtree('/kaggle/working/dataset')

selected_diseases = [
    'Acne and Rosacea Photos',
    'Eczema Photos',
    'Psoriasis pictures Lichen Planus and related diseases',
    'Tinea Ringworm Candidiasis and other Fungal Infections',
    'Warts Molluscum and other Viral Infections'
]

def clean_and_copy(file_paths, dst_folder, max_images=None):
    os.makedirs(dst_folder, exist_ok=True)
    random.seed(42)
    random.shuffle(file_paths)
    
    count = 0
    for src_path in file_paths:
        if max_images and count >= max_images:
            break
        try:
            with Image.open(src_path) as img:
                img.verify()
            filename = f"{count}_{os.path.basename(src_path)}"
            shutil.copy(src_path, os.path.join(dst_folder, filename))
            count += 1
        except Exception:
            pass
    return count

print("⏳ Filtering and cleaning the 5 skin diseases...")

for disease in selected_diseases:
    train_files = []
    test_files = []
    
    for root, dirs, files in os.walk('/kaggle/input'):
        if os.path.basename(root) == disease:
            imgs = [os.path.join(root, f) for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            if 'test' in root.lower():
                test_files.extend(imgs)
            else:
                train_files.extend(imgs)
                
    n_train = clean_and_copy(train_files, os.path.join(work_train_dir, disease), max_images=900)
    n_test = clean_and_copy(test_files, os.path.join(work_test_dir, disease), max_images=220)
    print(f"✔ {disease[:30]}... ---> Train: {n_train} | Test: {n_test}")

print("\n⏳ Searching and cleaning Healthy Skin images...")
healthy_images = []
for root, dirs, files in os.walk('/kaggle/input'):
    if 'normal' in root.lower() and 'dermnet' not in root.lower():
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                healthy_images.append(os.path.join(root, f))

if len(healthy_images) > 0:
    random.seed(42)
    random.shuffle(healthy_images)
    split_point = int(len(healthy_images) * 0.8)
    
    n_h_train = clean_and_copy(healthy_images[:split_point], os.path.join(work_train_dir, 'Healthy_Skin'), max_images=900)
    n_h_test = clean_and_copy(healthy_images[split_point:], os.path.join(work_test_dir, 'Healthy_Skin'), max_images=220)
    print(f"✔ Healthy_Skin ---> Train: {n_h_train} | Test: {n_h_test}")
else:
    print("❌ Normal skin folder not found!")

print("\n🎉 Done! Clean dataset is ready at: /kaggle/working/dataset")

⏳ Filtering and cleaning the 5 skin diseases...
✔ Acne and Rosacea Photos... ---> Train: 840 | Test: 220
✔ Eczema Photos... ---> Train: 900 | Test: 220
✔ Psoriasis pictures Lichen Plan... ---> Train: 900 | Test: 220
